def patch(a, b):
    return [a if b is None else patch(a,b)if isinstance(b,(dict,list))else b for a,b in zip_longest(a,b)]if isinstance(b,list)else{i:a if b.get(i)is None else patch(a,b[i])if isinstance(b[i],(dict,list))else b[i]for i,a in a.items()}if isinstance(b,dict)else b
def unjsonifysparsebytes(a, type, of=0):
    if isinstance(a, list):
        stype = np.dtype((type.base,type.shape[1:]))
        size = type.itemsize//(type.shape or[1])[0]
        for i,a in enumerate(a):
            if a is not None:
                yield from unjsonifysparsebytes(a, stype, of+i*size)
    elif isinstance(a, dict):
        for i,a in a.items():
            if a is not None:
                yield from unjsonifysparsebytes(a, (_:=type.fields[i])[0], of+_[1])
    else:
        yield of,type.type(a)
def tosparsebytes(a, of=0):
    if a.dtype.names:
        for i,(_,j) in a.dtype.fields.items():
            yield from tosparsebytes(a[i], of+j)
    elif a.ndim:
        yield from zip(sum(np.meshgrid(*[np.arange(0,i*a,a,'I')for i,a in zip(a.shape,a.strides)],copy=0,sparse=1,indexing='ij'),of).ravel(),a.flat)
    else:
        yield of,a
def joinbytes(a):
    if len(a)==1:
        return a[0]
    j,A = 0,bytearray(sum(a.nbytes for a in a))
    for a in a:
        A[j:j+a.nbytes] = a.data
        j += a.nbytes
    return A
def joinsparsebytes(a, max=256):
    I,j,A = 0,0,[]
    for i,a in a:
        assert i>=j
        if i>j or j-I+a.nbytes>max:
            if A:
                yield I,joinbytes(A)
            I = j = i
            A = []
        j += a.nbytes
        A.append(a)
    if A:
        yield I,joinbytes(A)

In [ ]:
import numpy as np
from itertools import zip_longest
def fromarrays(a, name):
    if len(a)<2:
        return a[0].view([(name[0],a[0].dtype)])
    i = min(next(i for i,(a,b)in enumerate(zip_longest(a[0].shape,b.shape+(0,)))if a!=b)for b in a[1:])
    b = np.empty(a[0].shape[:i], np.rec.format_parser([(a.dtype,a.shape[i:])for a in a],name,None))
    for name,a in zip(name,a):
        b[name] = a
    return b
def unpackbits(a):
    if a.dtype.base=='B':
        return np.unpackbits(a,-1).view('?')
    if a.dtype.names and any((_:=list(zip(*((_:=unpackbits(a[i]),_ is not a[i])for i in a.dtype.names))))[1]):
        return fromarrays(_[0], a.dtype.names)
    return a
def jsonify(a):
    return a.tolist()if not a.dtype.names else[jsonify(a)for a in a]if a.ndim else{i:jsonify(a[i])for i in a.dtype.names}
def unjsonify(a, type, df=0):
    return [unjsonify(a,type,df)for a in a]if isinstance(a,list)else tuple(unjsonify(a[i],type,df)if i in a else df for i,(type,_)in type.fields.items())if isinstance(a,dict)else a
def unjsonifysparse(a, type):
    if isinstance(a, list):
        stype = np.dtype((type.base,type.shape[1:]))
        size = type.itemsize//(type.shape or[1])[0]
        b,name,stype,of = zip(*[(a,str(i),a.dtype,i*size)for i,a in enumerate(a)if a is not None for a in[unjsonifysparse(a,stype)]])
        return np.array(b,None if all(a is not None for a in a)and all(i==stype[0]for i in stype[1:])else{'names':name,'formats':stype,'offsets':of}|({'itemsize':type.itemsize}if type else{}))
    if isinstance(a, dict):
        a,name,stype,of = zip(*[(b,i,(b.dtype,b.shape),of)for i,(type,of)in type.fields.items()for b in[a.get(i)]if b is not None for b in[unjsonifysparse(b,type)]])
        return np.array(a, {'names':name,'formats':stype,'offsets':of,'itemsize':type.itemsize})
    return type.type(a)
def tobytesparse(a):
    t = ~a[None].view('B')
    t.view(a.dtype)[:] = a
    a = a[None].view('B')
    t = (t==a).nonzero()[0]
    i = np.concat([[0],(t[1:]-t[:-1]>1).nonzero()[0]+1,[len(t)]])
    return [(t,a[t:t+i])for t,i in zip(t[i[:-1]],i[1:]-i[:-1])]
def slicesparse(a, i):
    a = a[::-1]
    for i,j in i:
        o = []
        while a:
            if len(_:=a[-1][1][max(i-a[-1][0],0):max(j-a[-1][0],0)]):
                o.append((max(a[-1][0]-i,0),_))
            if j-a[-1][0]<len(a[-1][1]):
                break
            a.pop()
        yield o
def frombytesparse(a, type):
    if type.shape:
        stype = np.dtype((type.base,type.shape[1:]))
        size = type.itemsize//type.shape[0]
        a,name,stype,of = zip(*[(a,str(i),(a.dtype,a.shape),i*size)for i,a in enumerate(slicesparse(a,zip(range(0,type.shape[0]*size,size),range(size,(type.shape[0]+1)*size,size))))if a for a in[frombytesparse(a,stype)]])
        return np.array(a, {'names':name,'formats':stype,'offsets':of,'itemsize':type.itemsize})
    if type.names:
        a,name,stype,of = zip(*[(a,i,(a.dtype,a.shape),of)for (i,(type,of)),a in zip(type.fields.items(),slicesparse(a,[(of,of+type.itemsize)for _,(type,of)in type.fields.items()]))if a for a in[frombytesparse(a,type)]])
        return np.array(a, {'names':name,'formats':stype,'offsets':of,'itemsize':type.itemsize})
    return np.frombuffer(a[0][1], type)[0]
def packbits(a):
    if a.dtype.names and a.dtype[0]=='?':
        return np.packbits(a[...,None].view('?'),-1)
    if a.dtype.names and any((_:=list(zip(*((_:=packbits(a[i]),_ is not a[i])for i in a.dtype.names))))[1]):
        return fromarrays(_[0], a.dtype.names)
    return a
def bytemask(a, t='B'):
    return a.filled((~a.data.view(t)).view(a.dtype)).view(t)==a.data.view(t)
def putmask(a, b, t='B'):
    return np.putmask(b.view(t), bytemask(a), a.view(t))
d = np.dtype('2B,f,2f')
D = np.dtype([
    ('flags',[
        ('a','?'),
        ('b','?'),
        ('c','?'),
        ('d','?'),
        ('e','?'),
        ('f','?'),
        ('g','?'),
        ('h','?'),
        ('i','?'),
        ('j','?'),
        ('k','?'),
        ('l','?'),
        ('m','?'),
        ('n','?'),
        ('o','?'),
        ('p','?')
    ]),
    ('val1','f'),
    ('val2','2f')
])
raw = bytearray(56)
a = unpackbits(np.frombuffer(raw,d)).view(D)